## Agent

In [ ]:
import random
from typing import Set, Tuple
from agents import Agent as AimaAgent, Thing as AimaThing, Direction


class Bump(AimaThing):
    def __init__(self, where: str):
        self.where = where


class Visited(AimaThing):
    def __init__(self, where: str):
        self.where = where


class Food(AimaThing):
    pass


class Water(AimaThing):
    pass


class SmartBlindDog(AimaAgent):
    def __init__(self, program):
        super().__init__(program)
        self.visited: Set[Tuple[int, ...]] = set()
        self.location: Tuple[int, int] = (0, 0)
        self.direction = Direction("down")
        self.visited.add(tuple(self.location))

    def moveforward(self, success: bool = True):
        if not success:
            return
        self.location = self.direction.move_forward(self.location)
        self.visited.add(tuple(self.location))

    def turn(self, d):
        self.direction = self.direction + d

    def eat(self, thing: AimaThing) -> bool:
        return isinstance(thing, Food)

    def drink(self, thing: AimaThing) -> bool:
        return isinstance(thing, Water)


ALL_DIRS = ('forward', 'left', 'right')
ACTION_MAP = {
    'forward': 'moveforward',
    'left': 'turnleft',
    'right': 'turnright',
}


def smart_program(percepts):
    blocked, walls = set(), set()

    for p in percepts:
        if isinstance(p, Food):
            return 'eat'
        if isinstance(p, Water):
            return 'drink'
        if isinstance(p, Bump):
            walls.add(p.where)
            blocked.add(p.where)
        elif isinstance(p, Visited):
            blocked.add(p.where)

    open_dirs = [d for d in ALL_DIRS if d not in blocked]

    if not open_dirs:
        open_dirs = [d for d in ALL_DIRS if d not in walls]

    if not open_dirs:
        return random.choice(['turnleft', 'turnright'])

    return ACTION_MAP[random.choice(open_dirs)]

## environment

In [ ]:
from overrides import override
from agents import GraphicEnvironment, Direction


class NoRepeatPark(GraphicEnvironment):
    @override
    def is_inbounds(self, location):
        x, y = location
        return 0 <= x < self.width and 0 <= y < self.height

    def percept(self, agent):
        things = self.list_things_at(agent.location)

        for where, heading in (
            ('forward', agent.direction),
            ('left', agent.direction + Direction.L),
            ('right', agent.direction + Direction.R)
        ):
            loc = heading.move_forward(agent.location)

            if not self.is_inbounds(loc):
                things.append(Bump(where))
            elif tuple(loc) in agent.visited:
                things.append(Visited(where))

        return things

    def execute_action(self, agent, action):
        if action == 'turnright':
            agent.turn(Direction.R)
        elif action == 'turnleft':
            agent.turn(Direction.L)
        elif action == 'moveforward':
            agent.moveforward()
        elif action in ('eat', 'drink'):
            target_class = Food if action == 'eat' else Water
            items = self.list_things_at(agent.location, tclass=target_class)
            if items:
                action_func = getattr(agent, action)
                if action_func(items[0]):
                    self.delete_thing(items[0])

    def is_done(self):
        no_edibles = not any(isinstance(t, (Food, Water)) for t in self.things)
        dead_agents = not any(a.is_alive() for a in self.agents)
        return dead_agents or no_edibles

### Simulation

In [ ]:
park = NoRepeatPark(
    5,
    5,
    color={
        'SmartBlindDog': (200, 0, 0),
        'Water': (0, 200, 200),
        'Food': (230, 115, 40),
    },
)
dog = SmartBlindDog(smart_program)
park.add_thing(dog, [0, 0])
park.add_thing(Food(), [3, 2])
park.add_thing(Water(), [2, 1])
park.run(100)

,,,,
,,,,
,,,,
,,,,
,,,,
